<h1>Aggregation with dissolve</h1>

<p>Spatial data are often more granular than needed. For example, you might have data on sub-national units, but you’re actually interested in studying patterns at the level of countries.</p>

<p>In a non-spatial setting, when you need summary statistics of the data, you can aggregate data using the
<a href="https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.groupby.html" title="(in pandas v2.3.0)"><code>groupby()</code></a> function. But for spatial data, you sometimes also need to aggregate geometric features. In the GeoPandas library, you can aggregate geometric features using the <a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.dissolve.html" title="geopandas.GeoDataFrame.dissolve"><code>dissolve()</code></a> function.</p>

<p><a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.dissolve.html" title="geopandas.GeoDataFrame.dissolve"><code>dissolve()</code></a> can be thought of as doing three things:</p>

<ol class="loweralpha simple">
<li><p>it dissolves all the geometries within a given group together into a single geometric feature (using the
<a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.union_all.html" title="geopandas.GeoSeries.union_all"><code>union_all()</code></a> method), and</p></li>
<li><p>it aggregates all the rows of data in a group using <a href="https://pandas.pydata.org/pandas-docs/stable/user_guide/groupby.html#groupby-aggregate" title="(in pandas v2.3.0)">groupby.aggregate</a>, and</p></li>
<li><p>it combines those two results.</p></li>
</ol>

# <h2><a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.dissolve.html" title="geopandas.GeoDataFrame.dissolve"><code>dissolve()</code></a> Example</h2>

<p>Take example of administrative areas in Nepal. You have districts, which are smaller, and zones, which are larger. A group of districts always compose a single zone. Suppose you are interested in Nepalese zone, but you only have Nepalese district-level data like the <cite>geoda.nepal</cite> dataset included in <cite>geodatasets</cite>. You can easily convert this to a zone-level dataset.</p>

<p>First, let’s look at the most simple case where you just want zone shapes and names.</p>

<div class="highlight-ipython notranslate"><div class="highlight" style="position: relative;"><span class="copybutton" title="Hide the prompts and output" data-hidden="false" style="cursor: pointer; position: absolute; top: 0px; right: 0px; font-family: monospace; padding-left: 0.2em; padding-right: 0.2em; border-radius: 0px 3px 0px 0px; user-select: none;">&gt;&gt;&gt;</span><pre tabindex="0"><span></span><span class="gp">In [1]: </span><span class="kn">import</span><span class="w"> </span><span class="nn">geodatasets</span>

<span class="gp">In [2]: </span><span class="n">nepal</span> <span class="o">=</span> <span class="n">geopandas</span><span class="o">.</span><span class="n">read_file</span><span class="p">(</span><span class="n">geodatasets</span><span class="o">.</span><span class="n">get_path</span><span class="p">(</span><span class="s1">'geoda.nepal'</span><span class="p">))</span>

<span class="gp">In [3]: </span><span class="n">nepal</span> <span class="o">=</span> <span class="n">nepal</span><span class="o">.</span><span class="n">rename</span><span class="p">(</span><span class="n">columns</span><span class="o">=</span><span class="p">{</span><span class="s2">"name_2"</span><span class="p">:</span> <span class="s2">"zone"</span><span class="p">})</span>  <span class="c1"># rename to remember the column</span>

<span class="gp">In [4]: </span><span class="n">nepal</span><span class="p">[[</span><span class="s2">"zone"</span><span class="p">,</span> <span class="s2">"geometry"</span><span class="p">]]</span><span class="o">.</span><span class="n">head</span><span class="p">()</span>
<span class="gh">Out[4]: </span>
<span class="go">          zone                                           geometry</span>
<span class="go">0  Dhaualagiri  POLYGON ((83.10834 28.6202, 83.1056 28.60976, ...</span>
<span class="go">1  Dhaualagiri  POLYGON ((83.99726 29.31675, 84 29.31576, 84 2...</span>
<span class="go">2  Dhaualagiri  POLYGON ((83.50688 28.79306, 83.51024 28.78809...</span>
<span class="go">3  Dhaualagiri  POLYGON ((83.70261 28.39837, 83.70435 28.39452...</span>
<span class="go">4      Bagmati  POLYGON ((85.52173 27.71822, 85.52359 27.71375...</span>
</pre>
</div>
</div>

<p>By default, <a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.dissolve.html" title="geopandas.GeoDataFrame.dissolve"><code>dissolve()</code></a> will pass <code>'first'</code> to <a href="https://pandas.pydata.org/pandas-docs/stable/user_guide/groupby.html#groupby-aggregate" title="(in pandas v2.3.0)">groupby.aggregate</a>.</p>

<div class="highlight-ipython notranslate"><div class="highlight" style="position: relative;"><span class="copybutton" title="Hide the prompts and output" data-hidden="false" style="cursor: pointer; position: absolute; top: 0px; right: 0px; font-family: monospace; padding-left: 0.2em; padding-right: 0.2em; border-radius: 0px 3px 0px 0px; user-select: none;">&gt;&gt;&gt;</span><pre tabindex="-1"><span></span><span class="gp">In [5]: </span><span class="n">nepal_zone</span> <span class="o">=</span> <span class="n">nepal</span><span class="p">[[</span><span class="s1">'zone'</span><span class="p">,</span> <span class="s1">'geometry'</span><span class="p">]]</span>

<span class="gp">In [6]: </span><span class="n">zones</span> <span class="o">=</span> <span class="n">nepal_zone</span><span class="o">.</span><span class="n">dissolve</span><span class="p">(</span><span class="n">by</span><span class="o">=</span><span class="s1">'zone'</span><span class="p">)</span>

<span class="gp">In [7]: </span><span class="n">zones</span><span class="o">.</span><span class="n">plot</span><span class="p">();</span>

<span class="gp">In [8]: </span><span class="n">zones</span><span class="o">.</span><span class="n">head</span><span class="p">()</span>
<span class="gh">Out[8]: </span>
<span class="go">                                                      geometry</span>
<span class="go">zone                                                          </span>
<span class="go">Bagmati      POLYGON ((85.87653 27.61234, 85.87355 27.60861...</span>
<span class="go">Bheri        POLYGON ((81.75089 28.31038, 81.75562 28.3074,...</span>
<span class="go">Dhaualagiri  POLYGON ((83.70647 28.39278, 83.70721 28.38781...</span>
<span class="go">Gandaki      POLYGON ((84.49995 28.74099, 84.50443 28.7441,...</span>
<span class="go">Janakpur     POLYGON ((86.26166 26.91417, 86.2588 26.91144,...</span>
</pre>
</div>
</div>

<img src="https://geopandas.org/en/stable/_images/zones1.png">

<p>If you are interested in aggregate populations, however, you can pass different functions to the
<a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.dissolve.html" title="geopandas.GeoDataFrame.dissolve"><code>dissolve()</code></a> method to aggregate populations using the <code>aggfunc =</code> argument:</p>

<div class="highlight-ipython notranslate"><div class="highlight" style="position: relative;"><span class="copybutton" title="Hide the prompts and output" data-hidden="false" style="cursor: pointer; position: absolute; top: 0px; right: 0px; font-family: monospace; padding-left: 0.2em; padding-right: 0.2em; border-radius: 0px 3px 0px 0px; user-select: none;">&gt;&gt;&gt;</span><pre tabindex="-1"><span></span><span class="gp">In [9]: </span><span class="n">nepal_pop</span> <span class="o">=</span> <span class="n">nepal</span><span class="p">[[</span><span class="s1">'zone'</span><span class="p">,</span> <span class="s1">'geometry'</span><span class="p">,</span> <span class="s1">'population'</span><span class="p">]]</span>

<span class="gp">In [10]: </span><span class="n">zones</span> <span class="o">=</span> <span class="n">nepal_pop</span><span class="o">.</span><span class="n">dissolve</span><span class="p">(</span><span class="n">by</span><span class="o">=</span><span class="s1">'zone'</span><span class="p">,</span> <span class="n">aggfunc</span><span class="o">=</span><span class="s1">'sum'</span><span class="p">)</span>

<span class="gp">In [11]: </span><span class="n">zones</span><span class="o">.</span><span class="n">plot</span><span class="p">(</span><span class="n">column</span> <span class="o">=</span> <span class="s1">'population'</span><span class="p">,</span> <span class="n">scheme</span><span class="o">=</span><span class="s1">'quantiles'</span><span class="p">,</span> <span class="n">cmap</span><span class="o">=</span><span class="s1">'YlOrRd'</span><span class="p">);</span>

<span class="gp">In [12]: </span><span class="n">zones</span><span class="o">.</span><span class="n">head</span><span class="p">()</span>
<span class="gh">Out[12]: </span>
<span class="go">                                                      geometry  population</span>
<span class="go">zone                                                                      </span>
<span class="go">Bagmati      POLYGON ((85.87653 27.61234, 85.87355 27.60861...     3750441</span>
<span class="go">Bheri        POLYGON ((81.75089 28.31038, 81.75562 28.3074,...     1463510</span>
<span class="go">Dhaualagiri  POLYGON ((83.70647 28.39278, 83.70721 28.38781...      516905</span>
<span class="go">Gandaki      POLYGON ((84.49995 28.74099, 84.50443 28.7441,...     1530310</span>
<span class="go">Janakpur     POLYGON ((86.26166 26.91417, 86.2588 26.91144,...     2818356</span>
</pre>
</div>
</div>

<img src="https://geopandas.org/en/stable/_images/zones2.png">

# <h2>Dissolve arguments</h2>

<p>The <code>aggfunc =</code> argument defaults to ‘first’ which means that the first row of attributes values found in the dissolve routine will be assigned to the resultant dissolved geodataframe.
However it also accepts other summary statistic options as allowed by
<a href="https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.groupby.html" title="(in pandas v2.3.0)"><code>pandas.groupby</code></a> including:</p>

<ul class="simple">
<li><p>‘first’</p></li>
<li><p>‘last’</p></li>
<li><p>‘min’</p></li>
<li><p>‘max’</p></li>
<li><p>‘sum’</p></li>
<li><p>‘mean’</p></li>
<li><p>‘median’</p></li>
<li><p>function</p></li>
<li><p>string function name</p></li>
<li><p>list of functions and/or function names, e.g. [np.sum, ‘mean’]</p></li>
<li><p>dict of axis labels -&gt; functions, function names or list of such.</p></li>
</ul>

<p>For example, to get the number of countries on each continent, as well as the populations of the largest and smallest country of each,
you can aggregate the <code>'name'</code> column using <code>'count'</code>, and the <code>'pop_est'</code> column using <code>'min'</code> and <code>'max'</code>:</p>

<div class="highlight-ipython notranslate"><div class="highlight" style="position: relative;"><span class="copybutton" title="Hide the prompts and output" data-hidden="false" style="cursor: pointer; position: absolute; top: 0px; right: 0px; font-family: monospace; padding-left: 0.2em; padding-right: 0.2em; border-radius: 0px 3px 0px 0px; user-select: none;">&gt;&gt;&gt;</span><pre tabindex="0"><span></span><span class="gp">In [13]: </span><span class="n">zones</span> <span class="o">=</span> <span class="n">nepal</span><span class="o">.</span><span class="n">dissolve</span><span class="p">(</span>
<span class="gp">   ....: </span>     <span class="n">by</span><span class="o">=</span><span class="s2">"zone"</span><span class="p">,</span>
<span class="gp">   ....: </span>     <span class="n">aggfunc</span><span class="o">=</span><span class="p">{</span>
<span class="gp">   ....: </span>         <span class="s2">"district"</span><span class="p">:</span> <span class="s2">"count"</span><span class="p">,</span>
<span class="gp">   ....: </span>         <span class="s2">"population"</span><span class="p">:</span> <span class="p">[</span><span class="s2">"min"</span><span class="p">,</span> <span class="s2">"max"</span><span class="p">],</span>
<span class="gp">   ....: </span>     <span class="p">},</span>
<span class="gp">   ....: </span> <span class="p">)</span>
<span class="gp">   ....: </span><span class="n">zones</span><span class="o">.</span><span class="n">head</span><span class="p">()</span>
<span class="gp">   ....: </span>
<span class="gh">Out[13]: </span>
<span class="go">                                                      geometry  ...  (population, max)</span>
<span class="go">zone                                                            ...                   </span>
<span class="go">Bagmati      POLYGON ((85.87653 27.61234, 85.87355 27.60861...  ...            1688131</span>
<span class="go">Bheri        POLYGON ((81.75089 28.31038, 81.75562 28.3074,...  ...             422812</span>
<span class="go">Dhaualagiri  POLYGON ((83.70647 28.39278, 83.70721 28.38781...  ...             250065</span>
<span class="go">Gandaki      POLYGON ((84.49995 28.74099, 84.50443 28.7441,...  ...             480851</span>
<span class="go">Janakpur     POLYGON ((86.26166 26.91417, 86.2588 26.91144,...  ...             765959</span>

<span class="go">[5 rows x 4 columns]</span>
</pre>
</div>
</div>